In [36]:
# Environment / path setup (kept minimal for agent execution)
from pathlib import Path
base_path = Path('.')
print('Base path set to current working directory.')

Base path set to current working directory.


# Insurance Analytics - Top 10 Business Questions

This notebook provides comprehensive analytics for key insurance business questions using actual policy and claims data.

## Table of Contents

| Question | Fabric Data Agent Question |
|----------|----------------------------|
| [Q1](#question-1-portfolio-mix-analysis) | What is the distribution of active policies by risk rating comparing 2023 vs 2024, and which risk segments show the largest percentage point changes? |
| [Q2](#question-2-pricing-alignment) | Are premium amounts monotonically increasing across risk ratings A through E, and what pricing inversions exist when adjusted for coverage amount? |
| [Q3](#question-3-coverage-adequacy) | Which 2024 policies have coverage-to-premium ratios more than 15% above their peer group median based on risk rating and age group? |
| [Q4](#question-4-product-mix-impact) | How do policy counts and average premiums by policy type compare between 2023 and 2024, and what are the market share changes? |
| [Q5](#question-5-aviation-analysis) | What are the differences in risk ratings, premiums, and claims performance between aviation-related occupations and non-aviation policies? |
| [Q6](#question-6-exam-exceptions) | How many high-risk (D/E rating) policies issued in 2024 did not require medical exams, and what are their characteristics? |
| [Q7](#question-7-occupation-anomalies) | Which occupations with at least 5 claims in 2024 have denial rates significantly different from the overall denial rate? |
| [Q8](#question-8-claims-trends) | What are the annual and quarterly trends in claim frequency and average claim amounts, particularly for 2024? |
| [Q9](#question-9-preexisting-conditions) | How do pre-existing conditions impact claim denial rates and average claim amounts compared to claims without pre-existing conditions? |
| [Q10](#question-10-data-quality) | What percentage of critical fields are missing in both policy and claims data, and how does this affect data completeness scores? |

---

**Data Sources:**
- `policy_portfolio.csv` - 890 policies with 24 fields (premiums, coverage, risk ratings, demographics)
- `claims_portfolio.csv` - 236 claims with 14 fields (amounts, status, causes, dates)

**Analysis Period:** 2020-2025 data with focus on 2024 YTD and 2023 comparisons

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
#import duckdb
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load Insurance Data
try:
    # Load policy and claims data
    policies_df = pd.read_csv('case_study/data/policy_portfolio.csv')
    claims_df = pd.read_csv('case_study/data/claims_portfolio.csv')
    
    print("Data loaded successfully!")
    print(f"Policies dataset shape: {policies_df.shape}")
    print(f"Claims dataset shape: {claims_df.shape}")
    
    # Convert date columns
    policies_df['issue_date'] = pd.to_datetime(policies_df['issue_date'])
    policies_df['last_premium_paid_date'] = pd.to_datetime(policies_df['last_premium_paid_date'])
    claims_df['claim_date'] = pd.to_datetime(claims_df['claim_date'])
    
    # Extract year and quarter information
    policies_df['issue_year'] = policies_df['issue_date'].dt.year
    policies_df['issue_quarter'] = policies_df['issue_date'].dt.quarter
    claims_df['claim_year'] = claims_df['claim_date'].dt.year
    claims_df['claim_quarter'] = claims_df['claim_date'].dt.quarter
    
    print("Date conversions completed!")
    
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    print("Please ensure the CSV files are in the correct location.")

Data loaded successfully!
Policies dataset shape: (890, 24)
Claims dataset shape: (236, 14)
Date conversions completed!


---

# Analysis Implementation

Each question below provides a complete analysis with business insights and recommendations.

## Question 1: Portfolio Mix Analysis {#question-1-portfolio-mix-analysis}

**Question:** What is the distribution of active policies by risk rating comparing 2023 vs 2024, and which risk segments show the largest percentage point changes?

**Python Code to Answer:**

In [39]:
# Question 1: Portfolio Mix Analysis
print("=" * 60)
print("QUESTION 1: PORTFOLIO MIX ANALYSIS")
print("=" * 60)

# Filter active policies for comparison years
active_2023 = policies_df[(policies_df['issue_year'] == 2023) & (policies_df['policy_status'] == 'Active')]
active_2024 = policies_df[(policies_df['issue_year'] == 2024) & (policies_df['policy_status'] == 'Active')]

print(f"\nData Overview:")
print(f"• Active policies issued in 2023: {len(active_2023):,}")
print(f"• Active policies issued in 2024: {len(active_2024):,}")

# Calculate risk distribution for both years
def calculate_risk_distribution(df, year):
    """Calculate risk rating distribution with percentages"""
    risk_dist = df.groupby('risk_rating').agg({
        'policy_id': 'count',
        'coverage_amount': 'sum'
    }).round(0)
    risk_dist.columns = ['Policy_Count', 'Total_Coverage']
    risk_dist['Policy_Pct'] = (risk_dist['Policy_Count'] / risk_dist['Policy_Count'].sum() * 100).round(1)
    risk_dist['Coverage_Pct'] = (risk_dist['Total_Coverage'] / risk_dist['Total_Coverage'].sum() * 100).round(1)
    return risk_dist

# Generate distributions
dist_2023 = calculate_risk_distribution(active_2023, 2023)
dist_2024 = calculate_risk_distribution(active_2024, 2024)

print(f"\n📊 2023 Risk Distribution:")
print(dist_2023)

print(f"\n📊 2024 Risk Distribution:")
print(dist_2024)

# Calculate percentage point changes
print(f"\n📈 Portfolio Mix Changes (2023 → 2024):")
all_risks = set(dist_2023.index) | set(dist_2024.index)
changes = []
for risk in sorted(all_risks):
    pct_2023 = dist_2023.loc[risk, 'Policy_Pct'] if risk in dist_2023.index else 0
    pct_2024 = dist_2024.loc[risk, 'Policy_Pct'] if risk in dist_2024.index else 0
    change = pct_2024 - pct_2023
    changes.append({'Risk_Rating': risk, '2023_Pct': pct_2023, '2024_Pct': pct_2024, 'Change_pp': change})

changes_df = pd.DataFrame(changes).sort_values('Change_pp', ascending=False)
print(changes_df)

# Key insights
largest_increase = changes_df.iloc[0]
largest_decrease = changes_df.iloc[-1]

print(f"\n🎯 Key Insights:")
print(f"• Largest increase: Risk {largest_increase['Risk_Rating']} (+{largest_increase['Change_pp']:.1f} percentage points)")
print(f"• Largest decrease: Risk {largest_decrease['Risk_Rating']} ({largest_decrease['Change_pp']:.1f} percentage points)")

# Coverage concentration
if len(active_2024) > 0:
    high_risk_coverage_2024 = active_2024[active_2024['risk_rating'].isin(['D', 'E'])]['coverage_amount'].sum()
    total_coverage_2024 = active_2024['coverage_amount'].sum()
    high_risk_pct = (high_risk_coverage_2024 / total_coverage_2024 * 100).round(1)
    print(f"• High-risk (D+E) coverage concentration in 2024: {high_risk_pct}%")

print(f"\n💡 Business Recommendation:")
print(f"Monitor portfolio drift toward higher-risk segments and ensure adequate pricing/reserves.")

QUESTION 1: PORTFOLIO MIX ANALYSIS

Data Overview:
• Active policies issued in 2023: 81
• Active policies issued in 2024: 12

📊 2023 Risk Distribution:
             Policy_Count  Total_Coverage  Policy_Pct  Coverage_Pct
risk_rating                                                        
A                      37      24340000.0        45.7          50.3
B                      15       6735000.0        18.5          13.9
C                       8       7040000.0         9.9          14.6
D                      18       9140000.0        22.2          18.9
E                       3       1110000.0         3.7           2.3

📊 2024 Risk Distribution:
             Policy_Count  Total_Coverage  Policy_Pct  Coverage_Pct
risk_rating                                                        
A                       2        580000.0        16.7           6.2
C                       4       3820000.0        33.3          40.9
D                       5       4050000.0        41.7          43.3
E    

**SQL Code to Answer:**

In [6]:
# Question 1: Portfolio Mix Analysis - SQL Implementation (Single Query)
import duckdb

# Create connection and load data into SQL tables
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 1: PORTFOLIO MIX ANALYSIS (SQL - SINGLE QUERY)")
print("=" * 60)

# Comprehensive SQL Query: All portfolio analysis in one query
comprehensive_query = """
WITH yearly_stats AS (
    -- Base statistics by year and risk rating
    SELECT 
        EXTRACT(YEAR FROM issue_date) as year,
        risk_rating,
        COUNT(*) as policy_count,
        SUM(coverage_amount) as total_coverage,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY EXTRACT(YEAR FROM issue_date)), 1) as policy_pct,
        ROUND(SUM(coverage_amount) * 100.0 / SUM(SUM(coverage_amount)) OVER (PARTITION BY EXTRACT(YEAR FROM issue_date)), 1) as coverage_pct
    FROM policies 
    WHERE EXTRACT(YEAR FROM issue_date) IN (2023, 2024) 
        AND policy_status = 'Active'
    GROUP BY EXTRACT(YEAR FROM issue_date), risk_rating
),
risk_comparison AS (
    -- Pivot data for year-over-year comparison
    SELECT 
        risk_rating,
        SUM(CASE WHEN year = 2023 THEN policy_count ELSE 0 END) as count_2023,
        SUM(CASE WHEN year = 2024 THEN policy_count ELSE 0 END) as count_2024,
        SUM(CASE WHEN year = 2023 THEN total_coverage ELSE 0 END) as coverage_2023,
        SUM(CASE WHEN year = 2024 THEN total_coverage ELSE 0 END) as coverage_2024,
        AVG(CASE WHEN year = 2023 THEN policy_pct END) as pct_2023,
        AVG(CASE WHEN year = 2024 THEN policy_pct END) as pct_2024
    FROM yearly_stats
    GROUP BY risk_rating
),
high_risk_stats AS (
    -- High-risk concentration for 2024
    SELECT 
        ROUND(
            SUM(CASE WHEN risk_rating IN ('D', 'E') THEN coverage_amount ELSE 0 END) * 100.0 / 
            SUM(coverage_amount), 1
        ) as high_risk_pct_2024
    FROM policies 
    WHERE EXTRACT(YEAR FROM issue_date) = 2024 AND policy_status = 'Active'
)
-- Final comprehensive result
SELECT 
    rc.risk_rating,
    rc.count_2023,
    rc.count_2024,
    (rc.count_2024 - rc.count_2023) as count_change,
    COALESCE(rc.pct_2023, 0) as pct_2023,
    COALESCE(rc.pct_2024, 0) as pct_2024,
    ROUND(COALESCE(rc.pct_2024, 0) - COALESCE(rc.pct_2023, 0), 1) as change_pp,
    ROUND(rc.coverage_2023, 0) as coverage_2023,
    ROUND(rc.coverage_2024, 0) as coverage_2024,
    hrs.high_risk_pct_2024
FROM risk_comparison rc
CROSS JOIN high_risk_stats hrs
WHERE rc.count_2023 > 0 OR rc.count_2024 > 0
ORDER BY change_pp DESC;
"""

print("\n📊 Comprehensive Portfolio Analysis (SQL):")
result = conn.execute(comprehensive_query).fetchdf()
print(result)

# Extract key insights from the single query result
if not result.empty:
    print(f"\n🎯 Key Insights from Single Query:")
    
    # Largest changes
    largest_increase = result.iloc[0]
    largest_decrease = result.iloc[-1]
    
    print(f"• Largest increase: Risk {largest_increase['risk_rating']} (+{largest_increase['change_pp']:.1f} percentage points)")
    print(f"• Largest decrease: Risk {largest_decrease['risk_rating']} ({largest_decrease['change_pp']:.1f} percentage points)")
    
    # High-risk concentration (same for all rows, so take first)
    high_risk_pct = result['high_risk_pct_2024'].iloc[0]
    print(f"• High-risk (D+E) coverage concentration in 2024: {high_risk_pct}%")

# Close connection
conn.close()

print(f"\n💡 Business Recommendation:")
print(f"SQL analysis confirms portfolio drift patterns. Monitor risk segment changes and adjust pricing/reserves accordingly.")

QUESTION 1: PORTFOLIO MIX ANALYSIS (SQL - SINGLE QUERY)

📊 Comprehensive Portfolio Analysis (SQL):
  risk_rating  count_2023  count_2024  count_change  pct_2023  pct_2024  \
0           C         8.0         4.0          -4.0       9.9      33.3   
1           D        18.0         5.0         -13.0      22.2      41.7   
2           E         3.0         1.0          -2.0       3.7       8.3   
3           B        15.0         0.0         -15.0      18.5       0.0   
4           A        37.0         2.0         -35.0      45.7      16.7   

   change_pp  coverage_2023  coverage_2024  high_risk_pct_2024  
0       23.4      7040000.0      3820000.0                52.9  
1       19.5      9140000.0      4050000.0                52.9  
2        4.6      1110000.0       900000.0                52.9  
3      -18.5      6735000.0            0.0                52.9  
4      -29.0     24340000.0       580000.0                52.9  

🎯 Key Insights from Single Query:
• Largest increase: Risk 

## Question 2: Pricing Alignment {#question-2-pricing-alignment}

**Question:** Are premium amounts monotonically increasing across risk ratings A through E, and what pricing inversions exist when adjusted for coverage amount?

**Python Code to Answer:**

In [40]:
# Question 2: Pricing Alignment
print("=" * 60)
print("QUESTION 2: PRICING ALIGNMENT")
print("=" * 60)

# Calculate pricing metrics by risk rating
pricing_stats = policies_df.groupby('risk_rating').agg({
    'premium': ['mean', 'median', 'std', 'count'],
    'coverage_amount': 'mean'
}).round(2)

# Flatten column names
pricing_stats.columns = ['Avg_Premium', 'Median_Premium', 'Std_Premium', 'Policy_Count', 'Avg_Coverage']

# Calculate risk-adjusted pricing (premium per $1000 coverage)
pricing_stats['Premium_per_1K'] = (pricing_stats['Avg_Premium'] / pricing_stats['Avg_Coverage'] * 1000).round(3)

print(f"\n📊 Pricing Analysis by Risk Rating:")
print(pricing_stats)

# Test monotonicity - premiums should increase with risk
risk_order = ['A', 'B', 'C', 'D', 'E']
available_risks = [r for r in risk_order if r in pricing_stats.index]

if len(available_risks) > 1:
    print(f"\n🔍 Monotonicity Test:")
    print(f"Risk progression: {' → '.join(available_risks)}")
    
    # Check raw premiums
    premiums = [pricing_stats.loc[risk, 'Avg_Premium'] for risk in available_risks]
    premium_monotonic = all(premiums[i] <= premiums[i+1] for i in range(len(premiums)-1))
    
    # Check risk-adjusted premiums
    risk_adj_premiums = [pricing_stats.loc[risk, 'Premium_per_1K'] for risk in available_risks]
    risk_adj_monotonic = all(risk_adj_premiums[i] <= risk_adj_premiums[i+1] for i in range(len(risk_adj_premiums)-1))
    
    print(f"\nRaw premiums: {' → '.join([f'${p:,.0f}' for p in premiums])}")
    print(f"Monotonic: {'✅ YES' if premium_monotonic else '❌ NO'}")
    
    print(f"\nRisk-adjusted (per $1K): {' → '.join([f'${p:.2f}' for p in risk_adj_premiums])}")
    print(f"Monotonic: {'✅ YES' if risk_adj_monotonic else '❌ NO'}")
    
    # Identify inversions
    inversions = []
    for i in range(len(available_risks)-1):
        current_risk = available_risks[i]
        next_risk = available_risks[i+1]
        if risk_adj_premiums[i] > risk_adj_premiums[i+1]:
            inversions.append(f"{current_risk} > {next_risk} (${risk_adj_premiums[i]:.2f} > ${risk_adj_premiums[i+1]:.2f} per $1K)")
    
    print(f"\n⚠️  Pricing Inversions Found:")
    if inversions:
        for inversion in inversions:
            print(f"• {inversion}")
    else:
        print("• None - pricing structure is properly aligned")

print(f"\n💡 Business Recommendation:")
if inversions:
    print(f"Review pricing for risk categories with inversions. Consider repricing {len(inversions)} identified issues.")
else:
    print(f"Pricing structure appears well-aligned with risk levels. Continue monitoring.")

QUESTION 2: PRICING ALIGNMENT

📊 Pricing Analysis by Risk Rating:
             Avg_Premium  Median_Premium  Std_Premium  Policy_Count  \
risk_rating                                                           
A                1908.95           662.0      3293.03           173   
B                4050.45          1455.1      7884.43           217   
C                2217.70          1245.0      2755.96           184   
D                1649.89          1111.5      2091.41           184   
E                1649.76          1382.5      1418.58           132   

             Avg_Coverage  Premium_per_1K  
risk_rating                                
A               867774.57           2.200  
B               997811.06           4.059  
C               628831.52           3.527  
D               384347.83           4.293  
E               285318.18           5.782  

🔍 Monotonicity Test:
Risk progression: A → B → C → D → E

Raw premiums: $1,909 → $4,050 → $2,218 → $1,650 → $1,650
Monotonic: ❌

**SQL Code to Answer:**

In [ ]:
# Question 2: Pricing Alignment - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 2: PRICING ALIGNMENT (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: All pricing analysis in one streamlined query
pricing_query = """
WITH pricing_stats AS (
    SELECT 
        risk_rating,
        COUNT(*) as policy_count,
        ROUND(AVG(premium), 2) as avg_premium,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY premium), 2) as median_premium,
        ROUND(AVG(coverage_amount), 2) as avg_coverage,
        ROUND(AVG(premium) / AVG(coverage_amount) * 1000, 3) as premium_per_1k
    FROM policies GROUP BY risk_rating
),
monotonicity AS (
    SELECT 
        p1.risk_rating || ' → ' || p2.risk_rating as progression,
        '$' || ROUND(p1.premium_per_1k, 2) || ' → $' || ROUND(p2.premium_per_1k, 2) || ' per $1K' as rate_progression,
        CASE WHEN p1.premium_per_1k <= p2.premium_per_1k THEN 'YES' ELSE 'NO' END as monotonic
    FROM pricing_stats p1 
    JOIN pricing_stats p2 ON 
        (p1.risk_rating = 'A' AND p2.risk_rating = 'B') OR
        (p1.risk_rating = 'B' AND p2.risk_rating = 'C') OR  
        (p1.risk_rating = 'C' AND p2.risk_rating = 'D') OR
        (p1.risk_rating = 'D' AND p2.risk_rating = 'E')
),
combined_results AS (
    SELECT 
        1 as sort_order,
        'STATS' as section, 
        risk_rating as detail, 
        policy_count, 
        avg_premium, 
        median_premium, 
        avg_coverage, 
        premium_per_1k,
        NULL as status
    FROM pricing_stats
    UNION ALL
    SELECT 
        2 as sort_order,
        'MONOTONICITY' as section,
        progression as detail,
        NULL, NULL, NULL, NULL, NULL,
        rate_progression || ' - ' || monotonic as status
    FROM monotonicity
    UNION ALL
    SELECT 
        3 as sort_order,
        'SUMMARY' as section,
        'Assessment' as detail,
        NULL, NULL, NULL, NULL, NULL,
        CASE WHEN (SELECT COUNT(*) FROM monotonicity WHERE monotonic = 'NO') > 0 
            THEN 'Found ' || (SELECT COUNT(*) FROM monotonicity WHERE monotonic = 'NO') || ' pricing inversions'
            ELSE 'No inversions - pricing aligned'
        END as status
)
SELECT section, detail, policy_count, avg_premium, median_premium, avg_coverage, premium_per_1k, status
FROM combined_results
ORDER BY sort_order, detail;
"""

print("\n📊 Comprehensive Pricing Analysis (SQL):")
result = conn.execute(pricing_query).fetchdf()

# Display results by section
stats = result[result['section'] == 'STATS']
mono = result[result['section'] == 'MONOTONICITY']
summary = result[result['section'] == 'SUMMARY']

print("\n📈 Pricing Statistics:")
print(stats[['detail', 'policy_count', 'avg_premium', 'median_premium', 'premium_per_1k']].to_string(index=False))

print("\n🔍 Monotonicity Check:")
for _, row in mono.iterrows():
    print(f"• {row['detail']}: {row['status']}")

print(f"\n⚠️  Assessment:")
print(f"• {summary['status'].iloc[0]}")

# Identify inversions
inversions = mono[mono['status'].str.contains('NO')]
if len(inversions) > 0:
    print(f"\n🚨 Pricing Inversions:")
    for _, row in inversions.iterrows():
        print(f"  - {row['detail']}")

conn.close()

print(f"\n💡 Business Recommendation:")
if len(inversions) > 0:
    print(f"Review {len(inversions)} pricing inversion(s) for risk-based repricing.")
else:
    print(f"Pricing structure is well-aligned with risk levels.")

QUESTION 2: PRICING ALIGNMENT (SQL - SINGLE QUERY)

📊 Comprehensive Pricing Analysis (SQL):


BinderException: Binder Error: Could not ORDER BY column "CASE  WHEN ((section = 'STATS')) THEN (1) WHEN ((section = 'MONOTONICITY')) THEN (2) WHEN ((section = 'SUMMARY')) THEN (3) ELSE NULL END": add the expression/function to every SELECT, or move the UNION into a FROM clause.

## Question 3: Coverage Adequacy {#question-3-coverage-adequacy}

**Question:** Which 2024 policies have coverage-to-premium ratios more than 15% above their peer group median based on risk rating and age group?

**Python Code to Answer:**

In [41]:
# Question 3: Coverage Adequacy
print("=" * 60)
print("QUESTION 3: COVERAGE ADEQUACY")
print("=" * 60)

# Focus on 2024 policies for adequacy analysis
policies_2024 = policies_df[policies_df['issue_year'] == 2024].copy()

if len(policies_2024) > 0:
    # Calculate coverage-to-premium ratio
    policies_2024['coverage_premium_ratio'] = policies_2024['coverage_amount'] / policies_2024['premium']
    
    # Create age groups for peer comparison
    policies_2024['age_group'] = pd.cut(policies_2024['age'], 
                                       bins=[0, 30, 40, 50, 60, 100], 
                                       labels=['<30', '30-39', '40-49', '50-59', '60+'])
    
    print(f"📊 Analyzing {len(policies_2024)} policies issued in 2024")
    
    # Calculate peer group medians
    peer_medians = policies_2024.groupby(['risk_rating', 'age_group'])['coverage_premium_ratio'].median()
    
    # Find outliers (>15% above peer median)
    outliers = []
    for idx, policy in policies_2024.iterrows():
        try:
            peer_median = peer_medians.loc[(policy['risk_rating'], policy['age_group'])]
            deviation_pct = ((policy['coverage_premium_ratio'] - peer_median) / peer_median * 100)
            
            if deviation_pct > 15:  # Significant outlier
                outliers.append({
                    'policy_id': policy['policy_id'],
                    'risk_rating': policy['risk_rating'],
                    'age_group': policy['age_group'],
                    'coverage_amount': policy['coverage_amount'],
                    'premium': policy['premium'],
                    'ratio': policy['coverage_premium_ratio'],
                    'peer_median': peer_median,
                    'deviation_pct': deviation_pct
                })
        except KeyError:
            continue  # Skip if no peer group
    
    print(f"\n🔍 Coverage Adequacy Analysis:")
    print(f"• Total policies analyzed: {len(policies_2024):,}")
    print(f"• Potential over-insurance cases (>15% above peers): {len(outliers)}")
    
    if outliers:
        outliers_df = pd.DataFrame(outliers)
        outliers_df = outliers_df.sort_values('deviation_pct', ascending=False)
        
        print(f"\n⚠️  Top Over-Insurance Cases:")
        top_cases = outliers_df.head(5)[['policy_id', 'risk_rating', 'coverage_amount', 'premium', 'deviation_pct']]
        print(top_cases.round(1))
        
        # Summary statistics
        total_excess_coverage = (outliers_df['coverage_amount'] - 
                               (outliers_df['peer_median'] * outliers_df['premium'])).sum()
        
        print(f"\n📈 Summary Statistics:")
        print(f"• Average deviation: {outliers_df['deviation_pct'].mean():.1f}%")
        print(f"• Maximum deviation: {outliers_df['deviation_pct'].max():.1f}%")
        print(f"• Total excess coverage amount: ${total_excess_coverage:,.0f}")
    
    else:
        print("• No significant over-insurance cases identified")
        
else:
    print("❌ No 2024 policies available for analysis")

print(f"\n💡 Business Recommendation:")
print(f"Review flagged cases for appropriate coverage levels and potential fraud indicators.")

QUESTION 3: COVERAGE ADEQUACY
📊 Analyzing 13 policies issued in 2024

🔍 Coverage Adequacy Analysis:
• Total policies analyzed: 13
• Potential over-insurance cases (>15% above peers): 1

⚠️  Top Over-Insurance Cases:
  policy_id risk_rating  coverage_amount  premium  deviation_pct
0  LIF-1225           A         400000.0    312.0           15.5

📈 Summary Statistics:
• Average deviation: 15.5%
• Maximum deviation: 15.5%
• Total excess coverage amount: $53,750

💡 Business Recommendation:
Review flagged cases for appropriate coverage levels and potential fraud indicators.


**SQL Code to Answer:**

In [ ]:
# Question 3: Coverage Adequacy - SQL Implementation
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 3: COVERAGE ADEQUACY (SQL)")
print("=" * 60)

# Comprehensive SQL Query for Coverage Adequacy Analysis
coverage_adequacy_query = """
WITH policies_2024 AS (
    -- Focus on 2024 policies with age groups
    SELECT 
        policy_id,
        risk_rating,
        CASE 
            WHEN age < 30 THEN '<30'
            WHEN age >= 30 AND age < 40 THEN '30-39'
            WHEN age >= 40 AND age < 50 THEN '40-49'
            WHEN age >= 50 AND age < 60 THEN '50-59'
            ELSE '60+'
        END as age_group,
        coverage_amount,
        premium,
        ROUND(coverage_amount / premium, 2) as coverage_premium_ratio
    FROM policies 
    WHERE EXTRACT(YEAR FROM issue_date) = 2024
),
peer_medians AS (
    -- Calculate peer group medians
    SELECT 
        risk_rating,
        age_group,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY coverage_premium_ratio) as peer_median
    FROM policies_2024
    GROUP BY risk_rating, age_group
),
outlier_analysis AS (
    -- Find policies with coverage ratios >15% above peer median
    SELECT 
        p.policy_id,
        p.risk_rating,
        p.age_group,
        p.coverage_amount,
        p.premium,
        p.coverage_premium_ratio,
        pm.peer_median,
        ROUND(((p.coverage_premium_ratio - pm.peer_median) / pm.peer_median * 100), 1) as deviation_pct
    FROM policies_2024 p
    LEFT JOIN peer_medians pm ON p.risk_rating = pm.risk_rating AND p.age_group = pm.age_group
    WHERE pm.peer_median IS NOT NULL 
        AND ((p.coverage_premium_ratio - pm.peer_median) / pm.peer_median * 100) > 15
),
summary_stats AS (
    -- Calculate summary statistics
    SELECT 
        COUNT(*) as total_outliers,
        ROUND(AVG(deviation_pct), 1) as avg_deviation,
        ROUND(MAX(deviation_pct), 1) as max_deviation,
        ROUND(SUM(coverage_amount - (peer_median * premium)), 0) as total_excess_coverage
    FROM outlier_analysis
)
-- Main results
SELECT 
    'OVERVIEW' as section,
    (SELECT COUNT(*) FROM policies_2024) as total_policies_2024,
    (SELECT COUNT(*) FROM outlier_analysis) as over_insurance_cases,
    NULL as policy_id,
    NULL as risk_rating,
    NULL as coverage_amount,
    NULL as premium,
    NULL as deviation_pct

UNION ALL

SELECT 
    'TOP_CASES' as section,
    NULL as total_policies_2024,
    NULL as over_insurance_cases,
    policy_id,
    risk_rating,
    coverage_amount,
    premium,
    deviation_pct
FROM outlier_analysis
ORDER BY deviation_pct DESC
LIMIT 5

UNION ALL

SELECT 
    'SUMMARY_STATS' as section,
    total_outliers,
    NULL as over_insurance_cases,
    NULL as policy_id,
    NULL as risk_rating,
    avg_deviation,
    max_deviation,
    total_excess_coverage
FROM summary_stats;
"""

print(f"\n📊 Analyzing 2024 policies for coverage adequacy (SQL):")
adequacy_result = conn.execute(coverage_adequacy_query).fetchdf()

# Display results by section
overview = adequacy_result[adequacy_result['section'] == 'OVERVIEW'].iloc[0]
top_cases = adequacy_result[adequacy_result['section'] == 'TOP_CASES']
summary = adequacy_result[adequacy_result['section'] == 'SUMMARY_STATS'].iloc[0]

print(f"\n🔍 Coverage Adequacy Analysis:")
print(f"• Total policies analyzed: {int(overview['total_policies_2024']):,}")
print(f"• Potential over-insurance cases (>15% above peers): {int(overview['over_insurance_cases'])}")

if int(overview['over_insurance_cases']) > 0:
    print(f"\n⚠️  Top Over-Insurance Cases (SQL):")
    top_cases_display = top_cases[['policy_id', 'risk_rating', 'coverage_amount', 'premium', 'deviation_pct']].copy()
    print(top_cases_display.to_string(index=False))
    
    print(f"\n📈 Summary Statistics (SQL):")
    print(f"• Average deviation: {summary['total_policies_2024']:.1f}%")
    print(f"• Maximum deviation: {summary['over_insurance_cases']:.1f}%")
    print(f"• Total excess coverage amount: ${summary['deviation_pct']:,.0f}")
else:
    print("• No significant over-insurance cases identified")

# Additional detailed analysis
detailed_query = """
SELECT 
    risk_rating,
    COUNT(*) as outlier_count,
    ROUND(AVG(deviation_pct), 1) as avg_deviation,
    ROUND(MAX(deviation_pct), 1) as max_deviation
FROM (
    SELECT 
        p.policy_id,
        p.risk_rating,
        p.coverage_premium_ratio,
        pm.peer_median,
        ROUND(((p.coverage_premium_ratio - pm.peer_median) / pm.peer_median * 100), 1) as deviation_pct
    FROM (
        SELECT 
            policy_id,
            risk_rating,
            CASE 
                WHEN age < 30 THEN '<30'
                WHEN age >= 30 AND age < 40 THEN '30-39'
                WHEN age >= 40 AND age < 50 THEN '40-49'
                WHEN age >= 50 AND age < 60 THEN '50-59'
                ELSE '60+'
            END as age_group,
            coverage_amount / premium as coverage_premium_ratio
        FROM policies 
        WHERE EXTRACT(YEAR FROM issue_date) = 2024
    ) p
    LEFT JOIN (
        SELECT 
            risk_rating,
            age_group,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY coverage_amount / premium) as peer_median
        FROM (
            SELECT 
                risk_rating,
                CASE 
                    WHEN age < 30 THEN '<30'
                    WHEN age >= 30 AND age < 40 THEN '30-39'
                    WHEN age >= 40 AND age < 50 THEN '40-49'
                    WHEN age >= 50 AND age < 60 THEN '50-59'
                    ELSE '60+'
                END as age_group,
                coverage_amount / premium as coverage_premium_ratio
            FROM policies 
            WHERE EXTRACT(YEAR FROM issue_date) = 2024
        ) sub
        GROUP BY risk_rating, age_group
    ) pm ON p.risk_rating = pm.risk_rating AND p.age_group = pm.age_group
    WHERE pm.peer_median IS NOT NULL 
        AND ((p.coverage_premium_ratio - pm.peer_median) / pm.peer_median * 100) > 15
) outliers
GROUP BY risk_rating
ORDER BY outlier_count DESC;
"""

print(f"\n📋 Outliers by Risk Rating (SQL):")
detailed_result = conn.execute(detailed_query).fetchdf()
if not detailed_result.empty:
    print(detailed_result.to_string(index=False))
else:
    print("No outliers found by risk rating")

# Close connection
conn.close()

print(f"\n💡 Business Recommendation (SQL Analysis):")
print(f"SQL analysis confirms coverage adequacy patterns. Review flagged cases for appropriate coverage levels and potential fraud indicators.")

## Question 4: Product Mix Impact {#question-4-product-mix-impact}

**Question:** How do policy counts and average premiums by policy type compare between 2023 and 2024, and what are the market share changes?

**Python Code to Answer:**

In [42]:
# Question 4: Product Mix Impact
print("=" * 60)
print("QUESTION 4: PRODUCT MIX IMPACT")
print("=" * 60)

# Analyze product performance by year
def analyze_product_mix(df, year):
    """Analyze product mix for a given year"""
    return df.groupby('policy_type').agg({
        'policy_id': 'count',
        'premium': 'mean',
        'coverage_amount': 'mean'
    }).round(0)

# Get product mix for both years
products_2023 = analyze_product_mix(policies_df[policies_df['issue_year'] == 2023], 2023)
products_2024 = analyze_product_mix(policies_df[policies_df['issue_year'] == 2024], 2024)

products_2023.columns = ['Count_2023', 'Avg_Premium_2023', 'Avg_Coverage_2023']
products_2024.columns = ['Count_2024', 'Avg_Premium_2024', 'Avg_Coverage_2024']

# Combine data
product_comparison = pd.concat([products_2023, products_2024], axis=1).fillna(0)

# Calculate changes
product_comparison['Count_Change'] = product_comparison['Count_2024'] - product_comparison['Count_2023']
product_comparison['Count_Change_Pct'] = ((product_comparison['Count_2024'] - product_comparison['Count_2023']) / 
                                         product_comparison['Count_2023'] * 100).round(1)

# Handle division by zero
product_comparison['Premium_Change_Pct'] = np.where(
    product_comparison['Avg_Premium_2023'] > 0,
    ((product_comparison['Avg_Premium_2024'] - product_comparison['Avg_Premium_2023']) / 
     product_comparison['Avg_Premium_2023'] * 100).round(1),
    np.nan
)

print(f"📊 Product Mix Comparison (2023 vs 2024):")
print(product_comparison[['Count_2023', 'Count_2024', 'Count_Change', 'Count_Change_Pct']])

print(f"\n💰 Premium Analysis:")
print(product_comparison[['Avg_Premium_2023', 'Avg_Premium_2024', 'Premium_Change_Pct']])

# Market share analysis
total_2023 = product_comparison['Count_2023'].sum()
total_2024 = product_comparison['Count_2024'].sum()

if total_2023 > 0 and total_2024 > 0:
    print(f"\n📈 Market Share Analysis:")
    print(f"Total policies - 2023: {total_2023:.0f}, 2024: {total_2024:.0f}")
    
    for product in product_comparison.index:
        share_2023 = (product_comparison.loc[product, 'Count_2023'] / total_2023 * 100).round(1)
        share_2024 = (product_comparison.loc[product, 'Count_2024'] / total_2024 * 100).round(1)
        share_change = share_2024 - share_2023
        print(f"• {product}: {share_2023}% → {share_2024}% ({share_change:+.1f}pp)")

# Identify trends
declining_products = product_comparison[product_comparison['Count_Change'] < 0].index.tolist()
growing_products = product_comparison[product_comparison['Count_Change'] > 0].index.tolist()

print(f"\n🎯 Key Insights:")
if growing_products:
    print(f"• Growing products: {', '.join(growing_products)}")
if declining_products:
    print(f"• Declining products: {', '.join(declining_products)}")

print(f"\n💡 Business Recommendation:")
print(f"Focus marketing efforts on growing segments and review strategy for declining products.")

QUESTION 4: PRODUCT MIX IMPACT
📊 Product Mix Comparison (2023 vs 2024):
                Count_2023  Count_2024  Count_Change  Count_Change_Pct
policy_type                                                           
Term Life               79        12.0         -67.0             -84.8
Universal Life           5         1.0          -4.0             -80.0
Whole Life               8         0.0          -8.0            -100.0

💰 Premium Analysis:
                Avg_Premium_2023  Avg_Premium_2024  Premium_Change_Pct
policy_type                                                           
Term Life                 1146.0            1613.0                40.8
Universal Life            4046.0             312.0               -92.3
Whole Life                1366.0               0.0              -100.0

📈 Market Share Analysis:
Total policies - 2023: 92, 2024: 13
• Term Life: 85.9% → 92.3% (+6.4pp)
• Universal Life: 5.4% → 7.7% (+2.3pp)
• Whole Life: 8.7% → 0.0% (-8.7pp)

🎯 Key Insights:
• Declin

## Question 5: Aviation Analysis {#question-5-aviation-analysis}

**Question:** What are the differences in risk ratings, premiums, and claims performance between aviation-related occupations and non-aviation policies?

**Python Code to Answer:**

In [43]:
# Question 5: Aviation Analysis
print("=" * 60)
print("QUESTION 5: AVIATION ANALYSIS")
print("=" * 60)

# Identify aviation-related occupations
aviation_keywords = ['pilot', 'aviation', 'aircraft', 'flight', 'airline']
aviation_mask = policies_df['occupation'].str.contains('|'.join(aviation_keywords), case=False, na=False)
policies_df['is_aviation'] = aviation_mask

aviation_count = aviation_mask.sum()
total_policies = len(policies_df)

print(f"📊 Aviation Segment Overview:")
print(f"• Aviation-related policies: {aviation_count:,} ({aviation_count/total_policies*100:.1f}% of portfolio)")
print(f"• Non-aviation policies: {total_policies - aviation_count:,}")

if aviation_count > 0:
    # Compare aviation vs non-aviation segments
    segment_analysis = policies_df.groupby('is_aviation').agg({
        'policy_id': 'count',
        'premium': 'mean',
        'coverage_amount': 'mean',
        'age': 'mean'
    }).round(0)
    
    segment_analysis.index = ['Non-Aviation', 'Aviation']
    segment_analysis.columns = ['Policy_Count', 'Avg_Premium', 'Avg_Coverage', 'Avg_Age']
    
    print(f"\n📈 Segment Comparison:")
    print(segment_analysis)
    
    # Risk distribution comparison
    risk_by_segment = pd.crosstab(policies_df['is_aviation'], policies_df['risk_rating'], normalize='index') * 100
    risk_by_segment.index = ['Non-Aviation (%)', 'Aviation (%)']
    
    print(f"\n🎯 Risk Rating Distribution:")
    print(risk_by_segment.round(1))
    
    # Claims analysis if possible
    if 'claim_id' in claims_df.columns:
        policy_claims = claims_df.merge(policies_df[['policy_id', 'is_aviation']], on='policy_id', how='left')
        
        claims_by_segment = policy_claims.groupby('is_aviation').agg({
            'claim_id': 'count',
            'claim_amount': 'mean',
            'claim_status': lambda x: (x == 'Approved').mean() * 100
        }).round(1)
        
        claims_by_segment.index = ['Non-Aviation', 'Aviation']
        claims_by_segment.columns = ['Total_Claims', 'Avg_Claim_Amount', 'Approval_Rate_Pct']
        
        print(f"\n💰 Claims Performance:")
        print(claims_by_segment)

print(f"\n💡 Business Recommendation:")
print(f"Monitor aviation segment for appropriate risk-based pricing and specialized underwriting.")

QUESTION 5: AVIATION ANALYSIS
📊 Aviation Segment Overview:
• Aviation-related policies: 48 (5.4% of portfolio)
• Non-aviation policies: 842

📈 Segment Comparison:
              Policy_Count  Avg_Premium  Avg_Coverage  Avg_Age
Non-Aviation           842       2383.0      633987.0     44.0
Aviation                48       2756.0     1185833.0     42.0

🎯 Risk Rating Distribution:
risk_rating          A     B     C     D     E
Non-Aviation (%)  20.5  25.8  19.8  19.4  14.5
Aviation (%)       0.0   0.0  35.4  43.8  20.8

💰 Claims Performance:
              Total_Claims  Avg_Claim_Amount  Approval_Rate_Pct
Non-Aviation           217          498359.4               94.5
Aviation                26         1064884.6               88.5

💡 Business Recommendation:
Monitor aviation segment for appropriate risk-based pricing and specialized underwriting.

📈 Segment Comparison:
              Policy_Count  Avg_Premium  Avg_Coverage  Avg_Age
Non-Aviation           842       2383.0      633987.0     4

## Question 6: Exam Exceptions {#question-6-exam-exceptions}

**Question:** How many high-risk (D/E rating) policies issued in 2024 did not require medical exams, and what are their characteristics?

**Python Code to Answer:**

In [45]:
# Question 6: Exam Exceptions
print("=" * 60)
print("QUESTION 6: EXAM EXCEPTIONS")
print("=" * 60)

# Focus on 2024 high-risk policies
high_risk_2024 = policies_df[
    (policies_df['issue_year'] == 2024) & 
    (policies_df['risk_rating'].isin(['D', 'E']))
]

print(f"📊 High-Risk Policy Analysis (2024):")
print(f"• Total high-risk (D/E) policies: {len(high_risk_2024):,}")

if len(high_risk_2024) > 0 and 'medical_exam_required' in policies_df.columns:
    # Analyze medical exam requirements
    exam_analysis = high_risk_2024.groupby(['risk_rating', 'medical_exam_required']).size().unstack(fill_value=0)
    print(f"\n🔍 Medical Exam Requirements:")
    print(exam_analysis)
    
    # Calculate exception rates
    exceptions = high_risk_2024[high_risk_2024['medical_exam_required'] == 'No']
    exception_rate = round((len(exceptions) / len(high_risk_2024) * 100), 1)
    
    print(f"\n⚠️  Exception Analysis:")
    print(f"• High-risk policies without medical exam: {len(exceptions):,}")
    print(f"• Exception rate: {exception_rate}%")
    
    if len(exceptions) > 0:
        print(f"\n📋 Exception Details:")
        exception_summary = exceptions.groupby('risk_rating').agg({
            'policy_id': 'count',
            'premium': 'mean',
            'coverage_amount': 'mean',
            'age': 'mean'
        }).round(0)
        print(exception_summary)
        
        print(f"\n🎯 Sample Exception Cases:")
        sample_cases = exceptions[['policy_id', 'risk_rating', 'age', 'premium', 'coverage_amount']].head(5)
        print(sample_cases)
    
    print(f"\n💡 Business Recommendation:")
    if exception_rate > 10:
        print(f"Exception rate of {exception_rate}% may be too high. Review underwriting guidelines.")
    else:
        print(f"Exception rate of {exception_rate}% appears reasonable. Continue monitoring.")
        
else:
    print("❌ Medical exam requirement data not available or no high-risk 2024 policies found")
    
    # Alternative analysis if medical_exam_required field is not available
    print(f"\n📋 Alternative Analysis (without medical exam field):")
    if len(high_risk_2024) > 0:
        risk_summary = high_risk_2024.groupby('risk_rating').agg({
            'policy_id': 'count',
            'premium': 'mean',
            'coverage_amount': 'mean',
            'age': 'mean'
        }).round(0)
        print(f"High-risk policy characteristics:")
        print(risk_summary)
        
        print(f"\n💡 Business Recommendation:")
        print(f"Consider implementing medical exam requirements for {len(high_risk_2024)} high-risk policies to improve risk assessment.")

QUESTION 6: EXAM EXCEPTIONS
📊 High-Risk Policy Analysis (2024):
• Total high-risk (D/E) policies: 7

🔍 Medical Exam Requirements:
medical_exam_required  No  Yes
risk_rating                   
D                       2    4
E                       0    1

⚠️  Exception Analysis:
• High-risk policies without medical exam: 2
• Exception rate: 28.6%

📋 Exception Details:
             policy_id  premium  coverage_amount   age
risk_rating                                           
D                    2    182.0          82500.0  28.0

🎯 Sample Exception Cases:
    policy_id risk_rating  age  premium  coverage_amount
198  LIF-1239           D   26    220.0         100000.0
870  LIF-1981           D   29    145.0          65000.0

💡 Business Recommendation:
Exception rate of 28.6% may be too high. Review underwriting guidelines.


## Question 7: Occupation Anomalies {#question-7-occupation-anomalies}

**Question:** Which occupations with at least 5 claims in 2024 have denial rates significantly different from the overall denial rate?

**Python Code to Answer:**

In [46]:
# Question 7: Occupation Anomalies
print("=" * 60)
print("QUESTION 7: OCCUPATION ANOMALIES")
print("=" * 60)

# Merge policies with claims for occupation analysis
policy_claims = claims_df.merge(policies_df[['policy_id', 'occupation']], on='policy_id', how='left')
claims_2024 = policy_claims[policy_claims['claim_year'] == 2024]

if len(claims_2024) > 0:
    # Calculate overall denial rate
    overall_denial_rate = (claims_2024['claim_status'] == 'Denied').mean()
    
    print(f"📊 Claims Analysis (2024):")
    print(f"• Total claims: {len(claims_2024):,}")
    print(f"• Overall denial rate: {overall_denial_rate:.1%}")
    
    # Analyze by occupation (minimum 5 claims)
    occupation_stats = claims_2024.groupby('occupation').agg({
        'claim_id': 'count',
        'claim_status': lambda x: (x == 'Denied').sum()
    })
    occupation_stats.columns = ['Total_Claims', 'Denied_Claims']
    occupation_stats['Denial_Rate'] = occupation_stats['Denied_Claims'] / occupation_stats['Total_Claims']
    
    # Filter occupations with ≥5 claims
    qualified_occupations = occupation_stats[occupation_stats['Total_Claims'] >= 5]
    
    print(f"\n🔍 Occupation Analysis (≥5 claims):")
    print(f"• Qualified occupations: {len(qualified_occupations)}")
    
    if len(qualified_occupations) > 0:
        # Find statistical anomalies (using simple threshold for demonstration)
        anomalies = qualified_occupations[
            (qualified_occupations['Denial_Rate'] > overall_denial_rate * 1.5) |  # 50% higher than average
            (qualified_occupations['Denial_Rate'] < overall_denial_rate * 0.5)    # 50% lower than average
        ]
        
        print(f"\n⚠️  Occupation Anomalies:")
        if len(anomalies) > 0:
            print(f"• Occupations with unusual denial rates: {len(anomalies)}")
            anomaly_summary = anomalies.sort_values('Denial_Rate', ascending=False)
            print(anomaly_summary.round(3))
        else:
            print(f"• No significant anomalies detected")
            
        print(f"\n📈 Top 10 Occupations by Claim Volume:")
        top_occupations = qualified_occupations.sort_values('Total_Claims', ascending=False).head(10)
        print(top_occupations.round(3))
        
else:
    print("❌ No 2024 claims data available for occupation analysis")

print(f"\n💡 Business Recommendation:")
print(f"Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.")

QUESTION 7: OCCUPATION ANOMALIES
📊 Claims Analysis (2024):
• Total claims: 87
• Overall denial rate: 0.0%

🔍 Occupation Analysis (≥5 claims):
• Qualified occupations: 3

⚠️  Occupation Anomalies:
• No significant anomalies detected

📈 Top 10 Occupations by Claim Volume:
                    Total_Claims  Denied_Claims  Denial_Rate
occupation                                                  
Restaurant Worker              6              0          0.0
Physical Therapist             5              0          0.0
Restaurant Manager             5              0          0.0

💡 Business Recommendation:
Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.


## Question 8: Claims Trends {#question-8-claims-trends}

**Question:** What are the annual and quarterly trends in claim frequency and average claim amounts, particularly for 2024?

**Python Code to Answer:**

In [47]:
# Question 8: Claims Trends
print("=" * 60)
print("QUESTION 8: CLAIMS TRENDS")
print("=" * 60)

# Analyze annual and quarterly trends
annual_trends = claims_df.groupby('claim_year').agg({
    'claim_id': 'count',
    'claim_amount': ['mean', 'sum']
}).round(0)

annual_trends.columns = ['Claim_Count', 'Avg_Amount', 'Total_Payout']

print(f"📊 Annual Claims Trends:")
print(annual_trends)

# 2024 quarterly analysis
if 2024 in claims_df['claim_year'].values:
    quarterly_2024 = claims_df[claims_df['claim_year'] == 2024].copy()
    quarterly_2024['quarter'] = quarterly_2024['claim_date'].dt.quarter
    
    quarterly_trends = quarterly_2024.groupby('quarter').agg({
        'claim_id': 'count',
        'claim_amount': 'mean'
    }).round(0)
    quarterly_trends.columns = ['Claim_Count', 'Avg_Amount']
    
    print(f"\n📈 2024 Quarterly Trends:")
    print(quarterly_trends)
    
    # Calculate trend direction
    if len(quarterly_trends) > 1:
        q1_avg = quarterly_trends.iloc[0]['Avg_Amount']
        latest_avg = quarterly_trends.iloc[-1]['Avg_Amount']
        trend_direction = "↗️ Increasing" if latest_avg > q1_avg else "↘️ Decreasing"
        change_pct = ((latest_avg - q1_avg) / q1_avg * 100).round(1)
        
        print(f"\n🎯 Trend Analysis:")
        print(f"• Direction: {trend_direction}")
        print(f"• Change from Q1: {change_pct:+.1f}%")

print(f"\n💡 Business Recommendation:")
print(f"Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.")

QUESTION 8: CLAIMS TRENDS
📊 Annual Claims Trends:
            Claim_Count  Avg_Amount  Total_Payout
claim_year                                       
2020                  1    110000.0      110000.0
2021                 12    283625.0     3403500.0
2022                 46    519380.0    23891500.0
2023                 93    593323.0    55179000.0
2024                 82    582500.0    47765000.0
2025                  2    650000.0     1300000.0

📈 2024 Quarterly Trends:
         Claim_Count  Avg_Amount
quarter                         
1                 48    616896.0
2                 31    543032.0
3                  3    440000.0

🎯 Trend Analysis:
• Direction: ↘️ Decreasing
• Change from Q1: -28.7%

💡 Business Recommendation:
Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.


## Question 9: Pre-existing Conditions {#question-9-preexisting-conditions}

**Question:** How do pre-existing conditions impact claim denial rates and average claim amounts compared to claims without pre-existing conditions?

**Python Code to Answer:**

In [48]:
# Question 9: Pre-existing Conditions Impact
print("=" * 60)
print("QUESTION 9: PRE-EXISTING CONDITIONS")
print("=" * 60)

if 'pre_existing_condition' in claims_df.columns:
    condition_analysis = claims_df.groupby('pre_existing_condition').agg({
        'claim_id': 'count',
        'claim_amount': 'mean',
        'claim_status': lambda x: (x == 'Denied').mean() * 100
    }).round(1)
    
    condition_analysis.columns = ['Total_Claims', 'Avg_Amount', 'Denial_Rate_Pct']
    condition_analysis.index = ['No Pre-existing', 'Has Pre-existing']
    
    print(f"📊 Pre-existing Condition Impact:")
    print(condition_analysis)
    
    # Calculate relative risk
    if len(condition_analysis) == 2:
        risk_ratio = (condition_analysis.loc['Has Pre-existing', 'Denial_Rate_Pct'] / 
                     condition_analysis.loc['No Pre-existing', 'Denial_Rate_Pct'])
        
        print(f"\n🎯 Risk Analysis:")
        print(f"• Denial rate - No condition: {condition_analysis.loc['No Pre-existing', 'Denial_Rate_Pct']:.1f}%")
        print(f"• Denial rate - Has condition: {condition_analysis.loc['Has Pre-existing', 'Denial_Rate_Pct']:.1f}%")
        print(f"• Relative risk: {risk_ratio:.2f}x higher with pre-existing conditions")
        
else:
    print("❌ Pre-existing condition data not available")

print(f"\n💡 Business Recommendation:")
print(f"Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.")

QUESTION 9: PRE-EXISTING CONDITIONS
📊 Pre-existing Condition Impact:
                  Total_Claims  Avg_Amount  Denial_Rate_Pct
No Pre-existing            169    668331.4              5.9
Has Pre-existing            67    279119.4              3.0

🎯 Risk Analysis:
• Denial rate - No condition: 5.9%
• Denial rate - Has condition: 3.0%
• Relative risk: 0.51x higher with pre-existing conditions

💡 Business Recommendation:
Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.


## Question 10: Data Quality {#question-10-data-quality}

**Question:** What percentage of critical fields are missing in both policy and claims data, and how does this affect data completeness scores?

**Python Code to Answer:**

In [49]:
# Question 10: Data Quality Assessment
print("=" * 60)
print("QUESTION 10: DATA QUALITY")
print("=" * 60)

print(f"📊 Data Quality Assessment:")

# Check for missing values in critical fields
critical_policy_fields = ['premium', 'coverage_amount', 'risk_rating', 'age']
critical_claims_fields = ['claim_amount', 'claim_status', 'claim_date']

print(f"\n🔍 Missing Data Analysis:")
print(f"Policy Data:")
for field in critical_policy_fields:
    if field in policies_df.columns:
        missing_count = policies_df[field].isna().sum()
        missing_pct = (missing_count / len(policies_df) * 100).round(1)
        print(f"• {field}: {missing_count:,} missing ({missing_pct}%)")

print(f"\nClaims Data:")
for field in critical_claims_fields:
    if field in claims_df.columns:
        missing_count = claims_df[field].isna().sum()
        missing_pct = (missing_count / len(claims_df) * 100).round(1)
        print(f"• {field}: {missing_count:,} missing ({missing_pct}%)")

# Data completeness score
total_policy_cells = len(policies_df) * len(critical_policy_fields)
total_claims_cells = len(claims_df) * len(critical_claims_fields)

policy_missing = sum(policies_df[field].isna().sum() for field in critical_policy_fields if field in policies_df.columns)
claims_missing = sum(claims_df[field].isna().sum() for field in critical_claims_fields if field in claims_df.columns)

policy_completeness = ((total_policy_cells - policy_missing) / total_policy_cells * 100).round(1)
claims_completeness = ((total_claims_cells - claims_missing) / total_claims_cells * 100).round(1)

print(f"\n📈 Overall Data Quality:")
print(f"• Policy data completeness: {policy_completeness}%")
print(f"• Claims data completeness: {claims_completeness}%")

quality_threshold = 95.0
if policy_completeness >= quality_threshold and claims_completeness >= quality_threshold:
    print(f"✅ Data quality meets standards (≥{quality_threshold}%)")
else:
    print(f"⚠️  Data quality below standards (<{quality_threshold}%)")

print(f"\n💡 Business Recommendation:")
print(f"Focus on improving data capture for fields with >5% missing values.")

print("\n" + "="*80)
print("📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED")
print("="*80)

QUESTION 10: DATA QUALITY
📊 Data Quality Assessment:

🔍 Missing Data Analysis:
Policy Data:
• premium: 0 missing (0.0%)
• coverage_amount: 0 missing (0.0%)
• risk_rating: 0 missing (0.0%)
• age: 0 missing (0.0%)

Claims Data:
• claim_amount: 0 missing (0.0%)
• claim_status: 0 missing (0.0%)
• claim_date: 0 missing (0.0%)

📈 Overall Data Quality:
• Policy data completeness: 100.0%
• Claims data completeness: 100.0%
✅ Data quality meets standards (≥95.0%)

💡 Business Recommendation:
Focus on improving data capture for fields with >5% missing values.

📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED
